# dr_evt 05: composite jobs, what dr_evt ships and the one gate we add

Our workload has composite jobs: several legs on different platforms that must all run or
none. dr_evt's C++ core knows nothing about them, but the repository above the core does.
It ships a composite-fragment file format, a coordination protocol that submits fragments
to several servers at the same logical time, an honest observation of whether every
fragment started, and two tests with an offline oracle. What it deliberately does not ship
is co-allocation: nothing reserves, cancels, or rolls back.

This session reads that machinery, runs it, reproduces it by hand, runs dr_evt's own test,
and then adds the single gate our market client uses, which makes partial starts
impossible in route mode. Everything the market does with composites is built on dr_evt's
conventions, not beside them.

Files to know:

| file | role |
|---|---|
| `python/examples/composite_jobs.csv`, `tests/test_traces/grpc/composite_jobs.csv` | the fragment format |
| `python/grpc_sync_coordinator.py` | the protocol and the `partial_start` observation |
| `tests/test_grpc_single_coordinator.py` | one client, two servers, four ordering cases, byte-compared to the CLI |
| `tests/test_grpc_multi_client_server.cpp`, `tests/run_grpc_composite_mpi_test.sh` | the same protocol with MPI barriers between paired clients |
| `docs/user-guide/client-server-use-cases.md`, `docs/TESTING_GUIDE.md` | the write-ups |

In [1]:
from pathlib import Path
import subprocess, socket, sys, os, json, tempfile
import pandas as pd

DR_EVT = Path.cwd().resolve()                        # run from learn/ or from the repository root
while not (DR_EVT / "CMakeLists.txt").exists() and DR_EVT != DR_EVT.parent:
    DR_EVT = DR_EVT.parent
assert (DR_EVT / "CMakeLists.txt").exists(), "run this notebook from learn/ inside a dr_evt checkout"
INSTALL = Path(os.environ.get("DR_EVT_INSTALL", DR_EVT / "install"))   # the cmake install prefix
OUT = Path.cwd() / "output"                                            # scratch space, gitignored
OUT.mkdir(exist_ok=True)
SERVER_BIN = INSTALL / "bin" / "dr_evt_server"
sys.path.insert(0, str(DR_EVT / "python"))
from grpc_multi_server import load_stubs, ServerSession, read_jobs
from grpc_sync_coordinator import read_systems, read_composites

grpc, pb, service, generated_dir = load_stubs(DR_EVT)

def free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0)); return s.getsockname()[1]

SERVER_DIR = Path(tempfile.mkdtemp(dir=OUT, prefix="composite_"))
ADDRESS = f"127.0.0.1:{free_port()}"
server = subprocess.Popen([str(SERVER_BIN), ADDRESS], cwd=SERVER_DIR,
                          stdout=(SERVER_DIR / "server.log").open("w"), stderr=subprocess.STDOUT)
ch = grpc.insecure_channel(ADDRESS); grpc.channel_ready_future(ch).result(timeout=15); ch.close()
print("one server on", ADDRESS, "; every platform below is a separate session on it")

one server on 127.0.0.1:64484 ; every platform below is a separate session on it


## 1. The format

One row per fragment. All fragments of a composite share a `submit_time`; composite
events are strictly time-ordered; a composite must name at least two systems. The
ordinary per-system traces use the plain simple format. `read_composites` groups the rows
and validates those rules (`python/grpc_sync_coordinator.py:56-95`); the C++ harness
enforces the same rules (`tests/test_grpc_multi_client_server.cpp:150-183`).

This is the wire format the market will emit for a composite decision: `composite_id` is
our job id, `system_id` is the platform, one row per leg.

In [2]:
EX = DR_EVT / "python" / "examples"
for name in ["sync_alpha_trace.csv", "sync_beta_trace.csv", "composite_jobs.csv"]:
    print(f"--- {name}\n{(EX / name).read_text()}")

systems = [{"system_id": "alpha"}, {"system_id": "beta"}]
events = read_composites(EX / "composite_jobs.csv", systems)
events

--- sync_alpha_trace.csv
job_submit_time,num_nodes,time_limit
0,20,100
10,20,100

--- sync_beta_trace.csv
job_submit_time,num_nodes,time_limit
0,40,100
10,30,100

--- composite_jobs.csv
composite_id,submit_time,system_id,num_nodes,q_id,time_limit
cross_site_1,25,alpha,30,1,120
cross_site_1,25,beta,40,1,120



[{'composite_id': 'cross_site_1',
  'submit_time': 25.0,
  'fragments': [{'system_id': 'alpha',
    'submit_time': 25.0,
    'num_nodes': 30,
    'queue': '1',
    'limit_time': 120.0},
   {'system_id': 'beta',
    'submit_time': 25.0,
    'num_nodes': 40,
    'queue': '1',
    'limit_time': 120.0}]}]

## 2. Run the shipped coordinator

`grpc_sync_coordinator.py` takes a systems file (`system_id,address,trace[,server_infile,
total_nodes]`) and the composite file. It pre-submits each system's ordinary trace, then
for each composite event: advances every system to the event time, snapshots, appends one
fragment per system, advances again, snapshots, and writes one JSON line. The shipped
example uses two servers; two sessions on one server are equivalent, so the systems file
below points both at our server. `server_infile` must be a path the server can read.

In [3]:
systems_csv = SERVER_DIR / "systems.csv"
systems_csv.write_text(
    "system_id,address,trace,server_infile,total_nodes\n"
    f"alpha,{ADDRESS},{EX / 'sync_alpha_trace.csv'},{EX / 'sync_alpha_trace.csv'},100\n"
    f"beta,{ADDRESS},{EX / 'sync_beta_trace.csv'},{EX / 'sync_beta_trace.csv'},100\n")
out = SERVER_DIR / "composite-results.jsonl"
r = subprocess.run([sys.executable, str(DR_EVT / "python" / "grpc_sync_coordinator.py"),
                    "--systems", str(systems_csv), "--composites", str(EX / "composite_jobs.csv"),
                    "--output", str(out)], capture_output=True, text=True)
print(r.stderr.strip())
record = json.loads(out.read_text())
pd.DataFrame(record["fragments"]).assign(composite_id=record["composite_id"], partial_start=record["partial_start"])

ordinary jobs pre-submitted: beta=2, alpha=2
cross_site_1 at t=25: PARTIAL_START


,system_id,requested_nodes,nodes_in_use_before,nodes_in_use_after,started_immediately,composite_id,partial_start
0,alpha,30,40,70,True,cross_site_1,True
1,beta,40,70,70,False,cross_site_1,True


`PARTIAL_START`. Alpha had 40 nodes busy (20 at t=0, 20 at t=10), so its 30-node fragment
started at once (40 to 70). Beta had 70 busy (40 at t=0, 30 at t=10); its 40-node fragment
needed more than the 30 free and joined beta's queue. The composite is now half running
and half waiting, which is exactly what all-or-nothing forbids. dr_evt reports it and
stops there: "the coordinator does not reserve, cancel, or roll back work on any server".

## 3. The protocol by hand

Six steps from `docs/user-guide/client-server-use-cases.md`, written out with
`ServerSession` so nothing is hidden. The step that matters is the second `AdvanceTo(tc)`:
appending only queues; the scheduler evaluates the same-time arrivals on the next advance.

In [4]:
HEADER_ONLY = OUT / "header_only.csv"
HEADER_ONLY.write_text("job_submit_time,num_nodes,queue,time_limit\n")

def open_session(name, total_nodes=100):
    s = ServerSession(ADDRESS, grpc, pb, service)
    s.call(pb.ClientMessage(init=pb.InitRequest(
        session_name=name, total_nodes=total_nodes, trace_format="simple", timestamp_format="epoch",
        backfill_policy="easy", priority_policy="fcfs", run_time_mode="limit", queue_impl="circular",
        msec_output=True, infile=str(HEADER_ONLY))))
    return s

def advance(s, t):  s.call(pb.ClientMessage(advance_to=pb.AdvanceToRequest(target_time=float(t))))
def stats(s):       return s.call(pb.ClientMessage(get_statistics=pb.GetStatisticsRequest())).get_statistics
def window(s):      return s.call(pb.ClientMessage(get_backfill_window=pb.GetBackfillWindowRequest())).get_backfill_window
def append(s, jobs): s.call(pb.ClientMessage(append_jobs=pb.AppendJobsRequest(requests=[pb.JobAppendData(**j) for j in jobs])))

def coordinator_step(sessions, event):
    tc = event["submit_time"]
    for s in sessions.values():                       # 1-2: barrier at tc, then snapshot
        advance(s, tc)
    before = {k: stats(s).nodes_in_use for k, s in sessions.items()}
    windows = {k: window(s) for k, s in sessions.items()}   # what each platform could take, before anything is appended
    for frag in event["fragments"]:                   # 3: one fragment per system at tc
        f = dict(frag); sid = f.pop("system_id")
        append(sessions[sid], [f])
    for s in sessions.values():                       # 4: evaluate the same-time arrivals
        advance(s, tc)
    after = {k: stats(s).nodes_in_use for k, s in sessions.items()}
    started = {f["system_id"]: after[f["system_id"]] - before[f["system_id"]] >= f["num_nodes"] for f in event["fragments"]}
    return dict(before=before, after=after, started=started,
                partial_start=any(started.values()) and not all(started.values()),
                windows={k: (w.available_nodes, w.shadow_time) for k, w in windows.items()})

sessions = {"alpha": open_session("hand-alpha"), "beta": open_session("hand-beta")}
append(sessions["alpha"], read_jobs(EX / "sync_alpha_trace.csv"))
append(sessions["beta"], read_jobs(EX / "sync_beta_trace.csv"))
result = coordinator_step(sessions, events[0])
print({k: v for k, v in result.items() if k != "windows"})
avail, shadow = result["windows"]["beta"]
print(f"beta's backfill window at t=25, before the fragment was appended: available={avail}, shadow={shadow}  "
      f"(shadow -1 = no head waiting; the fragment needs 40 > {avail} free, so it cannot start now)")
for s in sessions.values():
    s.call(pb.ClientMessage(finish_simulation=pb.FinishSimulationRequest())); s.close()

{'before': {'alpha': 40, 'beta': 70}, 'after': {'alpha': 70, 'beta': 70}, 'started': {'alpha': True, 'beta': False}, 'partial_start': True}
beta's backfill window at t=25, before the fragment was appended: available=30, shadow=-1.0  (shadow -1 = no head waiting; the fragment needs 40 > 30 free, so it cannot start now)


Same numbers as the shipped script. Note what beta's backfill window said before the
fragment was appended: no head waiting, 30 nodes available, request 40. That snapshot is
the information the coordinator ignores and the gate in section 5 uses.

## 4. dr_evt's own test and its oracle

`tests/test_grpc_single_coordinator.py` drives two servers through four ordering cases
(`tn = ta = tc`, `t0 = tc` with a second `AdvanceTo(tc)`, `tn = ta < tc`, `tc < t0`) and
then byte-compares each session's simulated and resource traces with an independent
`simulator` CLI run of that server's ordinary-plus-fragment stream. The oracle idea is
the one to copy for our tests: **a fragment is just a job on its platform**, so each
platform's schedule has an exact offline reference, and the cross-platform property is
checked separately. The MPI harness (`run_grpc_composite_mpi_test.sh`) does the same with
MPI barriers; it needs `mpirun`, which this Mac does not have.

In [5]:
for name in ["coordinator_server1.csv", "coordinator_server2.csv", "coordinator_composite_jobs.csv"]:
    print(f"--- tests/test_traces/grpc/{name}\n{(DR_EVT / 'tests' / 'test_traces' / 'grpc' / name).read_text()}")
r = subprocess.run([sys.executable, str(DR_EVT / "tests" / "test_grpc_single_coordinator.py"), str(SERVER_BIN)],
                   capture_output=True, text=True, cwd=DR_EVT)
print("test_grpc_single_coordinator.py exit code:", r.returncode, "(0 = every session matched the simulator oracle byte for byte)")
if r.returncode:
    print(r.stdout[-2000:], r.stderr[-2000:])

--- tests/test_traces/grpc/coordinator_server1.csv
job_submit_time,num_nodes,time_limit
0,10,20
10,10,15
10,1,1
20,10,10
30,10,10

--- tests/test_traces/grpc/coordinator_server2.csv
job_submit_time,num_nodes,time_limit
0,15,20
10,15,15
10,1,1
20,15,10
30,15,10

--- tests/test_traces/grpc/coordinator_composite_jobs.csv
composite_id,submit_time,system_id,num_nodes,time_limit
equal_boundary,10,server1,20,25
equal_boundary,10,server2,25,25
strict_boundary,25,server1,30,20
strict_boundary,25,server2,35,20



test_grpc_single_coordinator.py exit code: 0 (0 = every session matched the simulator oracle byte for byte)


## 5. The gate: `atomic_predicted_now`

dr_evt gives synchronized submission. The market adds one step between the barrier and
the append: every fragment must be able to start now on its platform, judged from that
platform's backfill window (notebook 02's rule). If any fragment cannot, nothing is
submitted and the composite waits for the next window.

Why this is a real reservation and not a guess: in route mode the controller is the only
thing submitting to any platform, it validates the whole window's demand against one
snapshot per platform, and dr_evt is deterministic. A fragment that fits free nodes on a
platform with no waiting head starts on the next `AdvanceTo` under EASY's head cascade.
`learn/probes/probe_composite.py` measures exactly that.

Below, the same composite event is offered at t=25 (deferred: beta cannot) and again at
t=100, when beta's 40-node job from t=0 has ended (admitted: both start together). The
begin times come from the session CSVs, not from a node-count delta.

In [6]:
def can_start_now(s, nodes, limit, now):
    w = window(s)
    if w.shadow_time < 0:
        return nodes <= w.available_nodes, "free_now" if nodes <= w.available_nodes else "does_not_fit"
    if nodes <= w.available_nodes and now + limit < w.shadow_time:
        return True, "backfill"
    return False, "behind_head"

def offer_composite(sessions, event, at):
    for s in sessions.values():
        advance(s, at)
    verdicts = {f["system_id"]: can_start_now(sessions[f["system_id"]], f["num_nodes"], f["limit_time"], at)
                for f in event["fragments"]}
    if not all(ok for ok, _ in verdicts.values()):
        return "deferred", verdicts
    for frag in event["fragments"]:
        f = dict(frag); sid = f.pop("system_id"); f["submit_time"] = at
        append(sessions[sid], [f])
    for s in sessions.values():
        advance(s, at)
    return "submitted", verdicts

sessions = {"alpha": open_session("gate-alpha"), "beta": open_session("gate-beta")}
append(sessions["alpha"], read_jobs(EX / "sync_alpha_trace.csv"))
append(sessions["beta"], read_jobs(EX / "sync_beta_trace.csv"))
for at in (25, 100):
    outcome, verdicts = offer_composite(sessions, events[0], at)
    print(f"t={at}: {outcome}  {verdicts}")

records = {}
for name, s in sessions.items():
    fin = s.call(pb.ClientMessage(finish_simulation=pb.FinishSimulationRequest())).finish_simulation
    s.close()
    df = pd.read_csv(SERVER_DIR / fin.simulated_trace_file)
    records[name] = df
    print(f"--- {name}\n{df.to_string(index=False)}")
starts = {name: float(df.iloc[-1].begin_time) for name, df in records.items()}   # the fragment is the last appended row
print("fragment begin times:", starts, "| partial_start:", len(set(starts.values())) != 1)

t=25: deferred  {'alpha': (True, 'free_now'), 'beta': (False, 'does_not_fit')}
t=100: submitted  {'alpha': (True, 'free_now'), 'beta': (True, 'free_now')}
--- alpha
 job_submit_time  begin_time  end_time  num_nodes  exit_status  time_limit
             0.0         0.0     100.0         20            0       100.0
            10.0        10.0     110.0         20            0       100.0
           100.0       100.0     220.0         30            0       120.0
--- beta
 job_submit_time  begin_time  end_time  num_nodes  exit_status  time_limit
             0.0         0.0     100.0         40            0       100.0
            10.0        10.0     110.0         30            0       100.0
           100.0       100.0     220.0         40            0       120.0
fragment begin times: {'alpha': 100.0, 'beta': 100.0} | partial_start: False


Both fragments began at t=100. The composite ran all-or-nothing without any change to
dr_evt, at the price of waiting for a window in which every platform could take its leg.
That price is the composite's wait time and it is measured, not hidden.

## 6. The three policies and where they live

| policy | what the controller does | what it is for |
|---|---|---|
| `atomic_predicted_now` | gate every fragment on its platform's backfill window; submit all or defer all; verify from records that all legs began at the window time; assert `partial_start_fraction == 0` | the headline experiment |
| `independent_fragments` | dr_evt's coordinator behavior: submit all fragments, observe `partial_start` | the diagnostic that shows what the gate prevents |
| `reject_composites` | never submit a composite | a baseline for mechanisms that cannot handle bundles |

Tests follow dr_evt's oracle: every platform's trace equals the `simulator` CLI run of
its ordinary-plus-fragment stream, plus the cross-platform equal-begin assertion. The
first fixture is this very example.

In [7]:
server.terminate(); server.wait(timeout=10)
generated_dir.cleanup()
print("server stopped; session files:", sorted(p.name for p in SERVER_DIR.glob("*.simulated.csv")))

server stopped; session files: ['alpha-1789412823094647-0-2865200908.simulated.csv', 'beta-1789412823094646-1-1120816688.simulated.csv', 'gate-alpha-1789412824994034-4-1523733490.simulated.csv', 'gate-beta-1789412824994880-5-840603266.simulated.csv', 'hand-alpha-1789412823140826-2-1257483290.simulated.csv', 'hand-beta-1789412823141923-3-1577134280.simulated.csv']


## Exercises

1. What does dr_evt's coordinator guarantee about a composite, and what does it not?
2. Why is the second `AdvanceTo(tc)` necessary, and what would you observe without it?
3. At t=25 beta's window said `available=30, shadow=-1`. Under which condition would a
   40-node fragment still be admissible on a platform with 30 free nodes?
4. Why is the gate exact in route mode but only a lower bound if a second submitter
   existed?
5. Describe the offline oracle dr_evt's test uses and how our composite test extends it.